# T14: RAG as a Subagent

In T13 one agent had the retrieval tool. Here the RAG agent itself becomes a tool of a **parent agent**. The parent handles the conversation and decides which tool to call; the RAG subagent handles retrieval and grounding. The parent never sees chunks, only the subagent's answer.

```
User ──► Parent agent ──► rag_agent tool ──► RAG subagent ──► retriever ──► vector store
              ├─────────► sum_two_numbers
              └─────────► answer directly
```

## Setup

Run T01 first: it builds the vector store at `backend/db/data_vs.db`. If the store is empty, `load_or_build_vector_store()` builds it from all documents, which takes several minutes.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, ask, load_or_build_vector_store
from ki_kompass_rag.tool_agents import (
    RAGAgentAsTool,
    RetrieveRelevantChunks,
    SumTwoNumbers,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)

## RAG subagent

The same agent as in T13, but it is only called for domain questions, so its prompt makes retrieval mandatory.

In [ ]:
RAG_PROMPT = "You answer questions about PrimePack AG. Always call retrieve_relevant_chunks before answering. Rely only on the retrieved chunks and cite their source files. Answer briefly."

rag_llm = OpenAILLM(tools=[RetrieveRelevantChunks(retriever)], tool_choice="auto")
rag_agent = ToolAgent(system_prompt=RAG_PROMPT, llm=rag_llm, max_steps=5)

## Parent agent

`RAGAgentAsTool` wraps the subagent as a tool: `call` runs `rag_agent.answer` and returns the answer text as `result` and the subagent's sources as `_sources`. The subagent gets no history, so the parent must send a self-contained question.

The parent also has `SumTwoNumbers`, so for each request it chooses: answer directly, add numbers, ask the subagent, or combine them.

In [ ]:
PARENT_PROMPT = "You are an assistant for PrimePack AG. Answer briefly. Use rag_agent for questions about PrimePack products, suppliers or documents, with one question per product, and keep the source files it cites. Use sum_two_numbers for additions."

parent_llm = OpenAILLM(
    tools=[RAGAgentAsTool(rag_agent), SumTwoNumbers()], tool_choice="auto"
)
parent_agent = ToolAgent(system_prompt=PARENT_PROMPT, llm=parent_llm, max_steps=5)

## Conversation

`ask` prints the source files collected from all subagent calls; `Sources: none` means the parent did not call the subagent. The third question needs both the subagent and the sum tool.

In [ ]:
history: list[LLMMessage] = []


async def chat(agent: ToolAgent, query: str) -> None:
    answer = await ask(agent, query, history)
    user_message = LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=query)]
    )
    history.extend([user_message, answer])

In [ ]:
await chat(
    parent_agent, "What is Einstein's theory of relativity? Answer in 2-3 sentences."
)

In [ ]:
await chat(
    parent_agent, "Which pallets in our portfolio have a third-party verified EPD?"
)

In [ ]:
await chat(
    parent_agent,
    "What is the total empty weight of a CPR wood pallet and a CPR recycled plastic pallet?",
)

In [ ]:
await chat(parent_agent, "Summarize our conversation, one line per message.")